# DSC LOGIKA UI 2026: v23 TabArena zoo screen

validation_v2 fold 0 (seed 42): each model is fitted by AutoGluon on the inner-fit rows with the inner 1/7 stop rows as tuning data, default TabArena hyperparameters; the outer fold is only predicted.

In [ ]:
%pip install -q polars==2.0.0 'tabarena[tabicl,tabdpt,limix,iltm,nori]==0.1.0' 'autogluon.tabular[mitra]>=1.6,<1.7'

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import polars as pl
FILES = {"features.py": "\"\"\"Leak-safe, customer-level feature builders (one selected application per customer).\n\nAll auxiliary tables are already truncated at decision_datetime (verified in 01_keys_target_time.py),\nbut every builder still filters on the cutoff so the code stays correct if that ever changes.\n\"\"\"\nimport polars as pl\n\nDAY = 86400.0\n\n\ndef _days(a, b):\n    return (pl.col(a) - pl.col(b)).dt.total_seconds() / DAY\n\n\ndef app_features(apps: pl.DataFrame, prof: pl.DataFrame) -> pl.DataFrame:\n    d = apps.join(prof, on=\"customer_id\", how=\"left\")\n    return d.select(\n        \"customer_id\",\n        \"loan_amount\", \"annual_interest_rate\", \"proposed_monthly_installment\", \"loan_purpose\",\n        \"age\", \"region\", \"employment_type\", \"employment_tenure_months\", \"declared_monthly_income\",\n        \"dependents\", \"housing_status\",\n        decision_lag_h=(pl.col(\"decision_datetime\") - pl.col(\"application_datetime\")).dt.total_seconds() / 3600,\n        app_hour=pl.col(\"application_datetime\").dt.hour(),\n        app_dow=pl.col(\"application_datetime\").dt.weekday(),\n        dti_declared=pl.col(\"proposed_monthly_installment\") / pl.col(\"declared_monthly_income\"),\n        lti_declared=pl.col(\"loan_amount\") / pl.col(\"declared_monthly_income\"),\n        tenure_over_age=pl.col(\"employment_tenure_months\") / (pl.col(\"age\") * 12),\n        income_per_head=pl.col(\"declared_monthly_income\") / (1 + pl.col(\"dependents\")),\n    )\n\n\ndef bureau_features(apps: pl.DataFrame, bureau: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", sel_app=pl.col(\"application_id\"), cutoff=pl.col(\"decision_datetime\"))\n    b = bureau.join(cut, on=\"customer_id\").filter(pl.col(\"available_datetime\") <= pl.col(\"cutoff\"))\n    b = b.with_columns(\n        stale_d=_days(\"cutoff\", \"report_datetime\"),\n        delay_d=_days(\"available_datetime\", \"report_datetime\"),\n        is_sel=pl.col(\"application_id\") == pl.col(\"sel_app\"),\n        prov=pl.col(\"bureau_provider\").str.replace(\"Bureau_\", \"\").str.to_lowercase(),\n    )\n    cur = b.filter(\"is_sel\")\n    num = [\"score\", \"active_accounts\", \"outstanding_balance\", \"revolving_credit_limit\",\n           \"revolving_outstanding_balance\", \"utilization\", \"inquiries_last_6m\", \"delinquent_accounts_last_12m\"]\n    agg = cur.group_by(\"customer_id\").agg(\n        bc_n_reports=pl.len(),\n        bc_n_scored=pl.col(\"score\").is_not_null().sum(),\n        bc_score_mean=pl.col(\"score\").mean(), bc_score_min=pl.col(\"score\").min(),\n        bc_score_max=pl.col(\"score\").max(), bc_score_std=pl.col(\"score\").std(),\n        bc_active_max=pl.col(\"active_accounts\").max(),\n        bc_outstanding_max=pl.col(\"outstanding_balance\").max(),\n        bc_revlimit_max=pl.col(\"revolving_credit_limit\").max(),\n        bc_revbal_max=pl.col(\"revolving_outstanding_balance\").max(),\n        bc_util_max=pl.col(\"utilization\").max(), bc_util_mean=pl.col(\"utilization\").mean(),\n        bc_inq_max=pl.col(\"inquiries_last_6m\").max(), bc_inq_mean=pl.col(\"inquiries_last_6m\").mean(),\n        bc_delinq_max=pl.col(\"delinquent_accounts_last_12m\").max(),\n        bc_delinq_sum=pl.col(\"delinquent_accounts_last_12m\").sum(),\n        bc_stale_mean=pl.col(\"stale_d\").mean(), bc_stale_max=pl.col(\"stale_d\").max(),\n        bc_delay_mean=pl.col(\"delay_d\").mean(),\n    )\n    piv = cur.pivot(on=\"prov\", index=\"customer_id\", values=[\"score\", \"utilization\", \"inquiries_last_6m\",\n                                                            \"delinquent_accounts_last_12m\", \"outstanding_balance\"],\n                    aggregate_function=\"first\")\n    piv = piv.rename({c: \"bp_\" + c.replace(\"inquiries_last_6m\", \"inq\").replace(\"delinquent_accounts_last_12m\", \"delinq\")\n                      .replace(\"outstanding_balance\", \"out\").replace(\"utilization\", \"util\")\n                      for c in piv.columns if c != \"customer_id\"})\n    # history: bureau pulled at earlier (previous) applications\n    hist = b.filter(~pl.col(\"is_sel\")).group_by(\"customer_id\").agg(\n        bh_n_pulls=pl.col(\"application_id\").n_unique(),\n        bh_score_mean=pl.col(\"score\").mean(), bh_score_min=pl.col(\"score\").min(),\n        bh_delinq_max=pl.col(\"delinquent_accounts_last_12m\").max(),\n        bh_inq_mean=pl.col(\"inquiries_last_6m\").mean(),\n        bh_last_score=pl.col(\"score\").sort_by(\"report_datetime\", \"bureau_observation_id\").drop_nulls().last(),\n        bh_last_outstanding=pl.col(\"outstanding_balance\").sort_by(\"report_datetime\", \"bureau_observation_id\").last(),\n        bh_pulls_180d=pl.col(\"application_id\").filter(pl.col(\"stale_d\") <= 180).n_unique(),\n    )\n    out = agg.join(piv, on=\"customer_id\", how=\"left\").join(hist, on=\"customer_id\", how=\"left\")\n    return out.with_columns(\n        bh_score_delta=pl.col(\"bc_score_mean\") - pl.col(\"bh_last_score\"),\n        bh_outstanding_delta=pl.col(\"bc_outstanding_max\") - pl.col(\"bh_last_outstanding\"),\n    )\n\n\ndef repayment_features(apps: pl.DataFrame, rep: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"))\n    r = rep.join(cut, on=\"customer_id\").filter(pl.col(\"due_date\") <= pl.col(\"cutoff\"))\n    r = r.with_columns(\n        age_d=_days(\"cutoff\", \"due_date\"),\n        unpaid=pl.col(\"amount_due\") - pl.col(\"amount_paid\"),\n        late_d=pl.when(pl.col(\"amount_paid\") >= pl.col(\"amount_due\")).then(_days(\"last_payment_datetime\", \"due_date\")),\n    )\n    w = lambda days: pl.col(\"age_d\") <= days  # noqa: E731\n    agg = r.group_by(\"customer_id\").agg(\n        rp_n_loans=pl.col(\"loan_id\").n_unique(), rp_n_inst=pl.len(),\n        rp_maxdpd_max=pl.col(\"max_days_past_due\").max(), rp_maxdpd_mean=pl.col(\"max_days_past_due\").mean(),\n        rp_n_dpd1=(pl.col(\"max_days_past_due\") > 0).sum(), rp_n_dpd30=(pl.col(\"max_days_past_due\") >= 30).sum(),\n        rp_n_dpd60=(pl.col(\"max_days_past_due\") >= 60).sum(), rp_n_dpd90=(pl.col(\"max_days_past_due\") >= 90).sum(),\n        rp_frac_dpd1=(pl.col(\"max_days_past_due\") > 0).mean(), rp_frac_dpd30=(pl.col(\"max_days_past_due\") >= 30).mean(),\n        rp_cur_dpd_max=pl.col(\"days_past_due\").max(), rp_n_unpaid=(pl.col(\"unpaid\") > 0).sum(),\n        rp_unpaid_amt=pl.col(\"unpaid\").sum(),\n        rp_paid_ratio=pl.col(\"amount_paid\").sum() / pl.col(\"amount_due\").sum(),\n        rp_late_mean=pl.col(\"late_d\").mean(), rp_late_max=pl.col(\"late_d\").max(),\n        rp_maxdpd_6m=pl.col(\"max_days_past_due\").filter(w(183)).max(),\n        rp_maxdpd_12m=pl.col(\"max_days_past_due\").filter(w(365)).max(),\n        rp_frac_dpd1_6m=(pl.col(\"max_days_past_due\") > 0).filter(w(183)).mean(),\n        rp_late_mean_6m=pl.col(\"late_d\").filter(w(183)).mean(),\n        rp_maxdpd_last3=pl.col(\"max_days_past_due\").sort_by(\"due_date\", \"loan_id\").tail(3).max(),\n        rp_days_since_delinq=pl.col(\"age_d\").filter(pl.col(\"max_days_past_due\") > 0).min(),\n        rp_days_since_due=pl.col(\"age_d\").min(),\n    )\n    # active loans: fewer installments observed than contracted term -> remaining obligation\n    loans = r.group_by(\"loan_id\").agg(customer_id=pl.col(\"customer_id\").first(), n_seen=pl.len(),\n                                      inst=pl.col(\"amount_due\").first())\n    loans = loans.join(prev.select(\"loan_id\", \"term_months\"), on=\"loan_id\", how=\"left\").with_columns(\n        remaining=(pl.col(\"term_months\") - pl.col(\"n_seen\")).clip(lower_bound=0))\n    act = loans.group_by(\"customer_id\").agg(\n        rp_n_active=(pl.col(\"remaining\") > 0).sum(),\n        rp_active_inst=pl.col(\"inst\").filter(pl.col(\"remaining\") > 0).sum(),\n        rp_remaining_oblig=(pl.col(\"remaining\") * pl.col(\"inst\")).sum(),\n    )\n    return agg.join(act, on=\"customer_id\", how=\"left\")\n\n\ndef account_features(apps: pl.DataFrame, acc: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"))\n    a = acc.join(cut, on=\"customer_id\").filter(\n        pl.col(\"month\").dt.offset_by(\"1mo\").cast(pl.Datetime) <= pl.col(\"cutoff\")).sort(\"customer_id\", \"month\")\n    a = a.with_columns(mago=pl.col(\"month\").rank(\"dense\", descending=True).over(\"customer_id\"))\n    l3 = pl.col(\"mago\") <= 3\n    out = a.group_by(\"customer_id\").agg(\n        ac_n_months=pl.len(),\n        ac_inflow_mean=pl.col(\"inflows\").mean(), ac_outflow_mean=pl.col(\"outflows\").mean(),\n        ac_income_mean=pl.col(\"income_like_credits\").mean(), ac_income_std=pl.col(\"income_like_credits\").std(),\n        ac_income_min=pl.col(\"income_like_credits\").min(),\n        ac_avgbal_mean=pl.col(\"average_balance\").mean(), ac_minbal_min=pl.col(\"minimum_balance\").min(),\n        ac_minbal_mean=pl.col(\"minimum_balance\").mean(),\n        ac_lowdays_mean=pl.col(\"low_balance_days\").mean(), ac_lowdays_max=pl.col(\"low_balance_days\").max(),\n        ac_frac_zero_min=(pl.col(\"minimum_balance\") == 0).mean(),\n        ac_net_flow_mean=(pl.col(\"inflows\") - pl.col(\"outflows\")).mean(),\n        ac_out_in_ratio=pl.col(\"outflows\").sum() / pl.col(\"inflows\").sum(),\n        ac_income_l3=pl.col(\"income_like_credits\").filter(l3).mean(),\n        ac_avgbal_l3=pl.col(\"average_balance\").filter(l3).mean(),\n        ac_lowdays_l3=pl.col(\"low_balance_days\").filter(l3).mean(),\n        ac_out_in_l3=pl.col(\"outflows\").filter(l3).sum() / pl.col(\"inflows\").filter(l3).sum(),\n        ac_last_close=pl.col(\"closing_balance\").last(),\n        ac_out_in_b=pl.col(\"outflows\").filter(pl.col(\"mago\").is_between(7, 12)).sum()\n        / pl.col(\"inflows\").filter(pl.col(\"mago\").is_between(7, 12)).sum(),\n        ac_lowdays_b=pl.col(\"low_balance_days\").filter(pl.col(\"mago\").is_between(7, 12)).mean(),\n        ac_avgbal_b=pl.col(\"average_balance\").filter(pl.col(\"mago\").is_between(7, 12)).mean(),\n        ac_out_in_l1=(pl.col(\"outflows\") / pl.col(\"inflows\")).filter(pl.col(\"mago\") == 1).first(),\n        ac_close_slope=pl.cov(pl.col(\"mago\").cast(pl.Float64), pl.col(\"closing_balance\").cast(pl.Float64))\n        / pl.col(\"mago\").cast(pl.Float64).var() * -1,\n    )\n    return out.with_columns(\n        ac_income_cv=pl.col(\"ac_income_std\") / pl.col(\"ac_income_mean\"),\n        ac_income_trend=pl.col(\"ac_income_l3\") / pl.col(\"ac_income_mean\"),\n        ac_bal_trend=pl.col(\"ac_avgbal_l3\") / (pl.col(\"ac_avgbal_mean\") + 1),\n        ac_out_in_delta=pl.col(\"ac_out_in_l3\") - pl.col(\"ac_out_in_b\"),\n        ac_lowdays_delta=pl.col(\"ac_lowdays_l3\") - pl.col(\"ac_lowdays_b\"),\n        ac_bal_delta=(pl.col(\"ac_avgbal_l3\") + 1) / (pl.col(\"ac_avgbal_b\") + 1),\n    )\n\n\ndef ecommerce_features(apps: pl.DataFrame, ec: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"))\n    e = ec.join(cut, on=\"customer_id\").filter(pl.col(\"transaction_datetime\") <= pl.col(\"cutoff\"))\n    e = e.with_columns(age_d=_days(\"cutoff\", \"transaction_datetime\"), done=pl.col(\"order_status\") == \"completed\")\n    # order level: order_total repeats on each item -> dedupe before aggregating\n    o = e.group_by(\"order_id\").agg(pl.col(\"customer_id\", \"age_d\", \"order_total\", \"payment_method\", \"done\").first())\n    agg = o.group_by(\"customer_id\").agg(\n        ec_n_orders=pl.len(), ec_cancel_rate=1 - pl.col(\"done\").mean(),\n        ec_spend=pl.col(\"order_total\").filter(\"done\").sum(),\n        ec_aov=pl.col(\"order_total\").filter(\"done\").mean(),\n        ec_recency_d=pl.col(\"age_d\").min(), ec_tenure_d=pl.col(\"age_d\").max(),\n        ec_orders_7d=(pl.col(\"age_d\") <= 7).sum(), ec_orders_14d=(pl.col(\"age_d\") <= 14).sum(),\n        ec_orders_30d=(pl.col(\"age_d\") <= 30).sum(), ec_orders_60d=(pl.col(\"age_d\") <= 60).sum(),\n        ec_orders_90d=(pl.col(\"age_d\") <= 90).sum(),\n        ec_base_wk=((pl.col(\"age_d\") > 120) & (pl.col(\"age_d\") <= 365)).sum()\n        / ((pl.col(\"age_d\").max().clip(upper_bound=365) - 120).clip(lower_bound=7) / 7),\n        ec_spend_30d=pl.col(\"order_total\").filter(pl.col(\"done\") & (pl.col(\"age_d\") <= 30)).sum(),\n        ec_spend_90d=pl.col(\"order_total\").filter(pl.col(\"done\") & (pl.col(\"age_d\") <= 90)).sum(),\n        ec_cancel_90d=(~pl.col(\"done\")).filter(pl.col(\"age_d\") <= 90).sum(),\n        **{f\"ec_pm_{m}\": (pl.col(\"payment_method\") == m).mean()\n           for m in [\"COD\", \"e_wallet\", \"bank_transfer\", \"debit_card\", \"credit_card\"]},\n    )\n    cat = (e.filter(\"done\").group_by(\"customer_id\", \"product_category\").agg(s=pl.col(\"item_total\").sum())\n           .with_columns(s=pl.col(\"s\") / pl.col(\"s\").sum().over(\"customer_id\"))\n           .pivot(on=\"product_category\", index=\"customer_id\", values=\"s\"))\n    cat = cat.rename({c: f\"ec_cat_{c}\" for c in cat.columns if c != \"customer_id\"})\n    out = agg.join(cat, on=\"customer_id\", how=\"left\").with_columns(\n        ec_spend_per_month=pl.col(\"ec_spend\") / (pl.col(\"ec_tenure_d\") / 30.4 + 1),\n        ec_spend_90d_ratio=pl.col(\"ec_spend_90d\") / 3 / (pl.col(\"ec_spend\") / (pl.col(\"ec_tenure_d\") / 30.4 + 1) + 1),\n    )\n    return out.with_columns(\n        pl.col(\"^ec_cat_.*$\").fill_null(0),\n        ec_accel_30d=(pl.col(\"ec_orders_30d\") / (30 / 7)) / (pl.col(\"ec_base_wk\") + 0.1),\n        ec_accel_90d=(pl.col(\"ec_orders_90d\") / (90 / 7)) / (pl.col(\"ec_base_wk\") + 0.1),\n    )\n\n\ndef prev_features(apps: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"), cur_rate=pl.col(\"annual_interest_rate\"),\n                      cur_amt=pl.col(\"loan_amount\"))\n    p = prev.join(cut, on=\"customer_id\").filter(pl.col(\"decision_datetime\") <= pl.col(\"cutoff\"))\n    p = p.with_columns(age_d=_days(\"cutoff\", \"application_datetime\"))\n    out = p.group_by(\"customer_id\").agg(\n        pv_n=pl.len(), pv_n_appr=(pl.col(\"decision\") == \"approved\").sum(),\n        pv_n_rej=(pl.col(\"decision\") == \"rejected\").sum(), pv_n_canc=(pl.col(\"decision\") == \"cancelled\").sum(),\n        pv_rej_rate=(pl.col(\"decision\") == \"rejected\").mean(),\n        pv_days_since_last=pl.col(\"age_d\").min(), pv_days_since_first=pl.col(\"age_d\").max(),\n        pv_n_90d=(pl.col(\"age_d\") <= 90).sum(), pv_n_180d=(pl.col(\"age_d\") <= 180).sum(),\n        pv_rej_365d=((pl.col(\"decision\") == \"rejected\") & (pl.col(\"age_d\") <= 365)).sum(),\n        pv_rate_mean=pl.col(\"annual_interest_rate\").mean(), pv_rate_max=pl.col(\"annual_interest_rate\").max(),\n        pv_rate_last=pl.col(\"annual_interest_rate\").sort_by(\"application_datetime\", \"application_id\").drop_nulls().last(),\n        pv_appr_ratio=(pl.col(\"approved_amount\") / pl.col(\"requested_amount\")).mean(),\n        pv_req_mean=pl.col(\"requested_amount\").mean(), pv_appr_sum=pl.col(\"approved_amount\").sum(),\n        pv_appr_max=pl.col(\"approved_amount\").max(),\n        pv_last_decision_rej=(pl.col(\"decision\").sort_by(\"application_datetime\", \"application_id\").last() == \"rejected\"),\n        cur_rate=pl.col(\"cur_rate\").first(), cur_amt=pl.col(\"cur_amt\").first(),\n    )\n    return out.with_columns(\n        pv_rate_delta=pl.col(\"cur_rate\") - pl.col(\"pv_rate_last\"),\n        pv_amt_vs_max=pl.col(\"cur_amt\") / pl.col(\"pv_appr_max\"),\n        pv_last_decision_rej=pl.col(\"pv_last_decision_rej\").cast(pl.Int8),\n    ).drop(\"cur_rate\", \"cur_amt\")\n\n\ndef v2_features(apps: pl.DataFrame, tables: dict) -> pl.DataFrame:\n    \"\"\"v2 additions (eda/21-26): signals left in the v1 residual. Same cutoff rules as the v1 builders.\"\"\"\n    cut = apps.select(\"customer_id\", \"application_id\", cutoff=pl.col(\"decision_datetime\"),\n                      inst_new=pl.col(\"proposed_monthly_installment\"), amt_new=pl.col(\"loan_amount\"))\n    # bureau: score trend/range over every pull, newest report, delinquency over all pulls, external inquiries\n    b = tables[\"bureau_observations\"].join(cut.select(\"customer_id\", \"cutoff\", sel=pl.col(\"application_id\")), on=\"customer_id\")\n    b = b.filter(pl.col(\"available_datetime\") <= pl.col(\"cutoff\")).with_columns(\n        age_d=_days(\"cutoff\", \"report_datetime\"), is_sel=pl.col(\"application_id\") == pl.col(\"sel\"))\n    bz = b.group_by(\"customer_id\").agg(\n        bz_trend=pl.cov(pl.col(\"age_d\"), pl.col(\"score\")) / pl.col(\"age_d\").var() * -365,\n        bz_range=pl.col(\"score\").max() - pl.col(\"score\").min(),\n        bz_newest=pl.col(\"score\").sort_by(\"report_datetime\", \"bureau_observation_id\").drop_nulls().last(),\n        bz_delinq_all_mean=pl.col(\"delinquent_accounts_last_12m\").mean(),\n        bz_active_trend=pl.col(\"active_accounts\").filter(\"is_sel\").mean()\n        - pl.col(\"active_accounts\").filter(~pl.col(\"is_sel\")).mean(),\n        bz_inq_cur=pl.col(\"inquiries_last_6m\").filter(\"is_sel\").mean(),\n    )\n    p = tables[\"previous_applications\"].join(cut, on=\"customer_id\").filter(pl.col(\"decision_datetime\") <= pl.col(\"cutoff\"))\n    own6 = p.filter(_days(\"cutoff\", \"application_datetime\") <= 183).group_by(\"customer_id\").agg(own6=pl.len())\n    bz = bz.join(own6, on=\"customer_id\", how=\"left\").with_columns(\n        bz_inq_external=pl.col(\"bz_inq_cur\") - pl.col(\"own6\").fill_null(0)).drop(\"bz_inq_cur\", \"own6\")\n    # previous approvals: tenor mix, recency, recent approved volume, remaining scheduled obligation\n    ap = p.filter(pl.col(\"decision\") == \"approved\").with_columns(\n        age_d=_days(\"cutoff\", \"decision_datetime\"),\n        rem=(pl.col(\"term_months\") - (_days(\"cutoff\", \"decision_datetime\") / 30.44).floor()).clip(lower_bound=0))\n    ob = ap.group_by(\"customer_id\").agg(\n        ob_term_mean=pl.col(\"term_months\").mean(), ob_days_since_appr=pl.col(\"age_d\").min(),\n        ob_amt_90d=pl.col(\"approved_amount\").filter(pl.col(\"age_d\") <= 90).sum(),\n        ob_remaining=(pl.col(\"scheduled_monthly_installment\") * pl.col(\"rem\")).sum(),\n        amt_new=pl.col(\"amt_new\").first(),\n    ).with_columns(ob_amt_vs_last=pl.col(\"amt_new\") / (pl.col(\"ob_amt_90d\") + 1e6)).drop(\"amt_new\")\n    # repayment: first installment, latest loan, 1-29 dpd share, recent trend, streaks, severity when late\n    r = tables[\"repayment_history\"].join(cut.select(\"customer_id\", \"cutoff\"), on=\"customer_id\")\n    # loan_id breaks due_date ties between concurrent loans -> deterministic order\n    r = r.filter(pl.col(\"due_date\") <= pl.col(\"cutoff\")).sort(\"customer_id\", \"due_date\", \"loan_id\").with_columns(\n        late=(pl.col(\"max_days_past_due\") > 0).cast(pl.Int32),\n        paid=pl.col(\"amount_paid\") >= pl.col(\"amount_due\"),\n        late_d=_days(\"last_payment_datetime\", \"due_date\"),\n        k=pl.col(\"due_date\").rank(\"ordinal\").over(\"loan_id\"),\n        loan_start=pl.col(\"due_date\").min().over(\"loan_id\"))\n    # streak of consecutive late installments within one loan (concurrent loans interleave in time)\n    r = r.with_columns(run=(pl.col(\"late\") != pl.col(\"late\").shift(1)).fill_null(True).cum_sum().over(\"loan_id\"))\n    streak = (r.filter(pl.col(\"late\") == 1).group_by(\"customer_id\", \"loan_id\", \"run\").agg(n=pl.len())\n              .group_by(\"customer_id\").agg(rq_streak_max=pl.col(\"n\").max()))\n    last = (r.filter(pl.col(\"loan_start\") == pl.col(\"loan_start\").max().over(\"customer_id\"))\n            .group_by(\"customer_id\").agg(rq_lastloan_late=pl.col(\"late\").mean(), rq_lastloan_maxdpd=pl.col(\"max_days_past_due\").max()))\n    dpd, late = pl.col(\"max_days_past_due\"), pl.col(\"max_days_past_due\") > 0\n    rq = r.group_by(\"customer_id\").agg(\n        rq_first_inst_late=pl.col(\"late\").filter(pl.col(\"k\") == 1).mean(),\n        rq_frac_1_29=(late & (dpd < 30)).mean(),\n        rq_late_trend=pl.col(\"late\").tail(6).mean() - pl.col(\"late\").head(pl.len() - 6).mean(),\n        sv_maxdpd_mean_late=dpd.filter(late).mean(),\n        sv_late_d_mean=pl.col(\"late_d\").filter(pl.col(\"paid\") & (pl.col(\"late_d\") > 0)).mean(),\n        sv_share30_of_late=(dpd >= 30).sum() / late.sum(),\n        sv_unpaid_of_late=(~pl.col(\"paid\") & late).sum() / late.sum(),\n    ).join(streak, on=\"customer_id\", how=\"left\").join(last, on=\"customer_id\", how=\"left\")\n    # account: non-income inflows, recent outflows, worst free cash flow, new installment vs free cash / min balance\n    a = tables[\"account_behavior\"].join(cut.select(\"customer_id\", \"cutoff\", \"inst_new\"), on=\"customer_id\")\n    a = a.filter(pl.col(\"month\").dt.offset_by(\"1mo\").cast(pl.Datetime) <= pl.col(\"cutoff\")).with_columns(\n        l3=pl.col(\"month\").rank(\"dense\", descending=True).over(\"customer_id\") <= 3,\n        free=pl.col(\"inflows\") - pl.col(\"outflows\"))\n    cf = a.group_by(\"customer_id\").agg(\n        cf_nonincome_share=1 - pl.col(\"income_like_credits\").sum() / pl.col(\"inflows\").sum(),\n        cf_nonincome_l3=1 - pl.col(\"income_like_credits\").filter(\"l3\").sum() / pl.col(\"inflows\").filter(\"l3\").sum(),\n        cf_out_l3=pl.col(\"outflows\").filter(\"l3\").mean(), cf_free_min=pl.col(\"free\").min(),\n        cf_free_l3=pl.col(\"free\").filter(\"l3\").mean(), cf_inc_l3=pl.col(\"income_like_credits\").filter(\"l3\").mean(),\n        cf_minbal_l3=pl.col(\"minimum_balance\").filter(\"l3\").min(), inst_new=pl.col(\"inst_new\").first(),\n    ).with_columns(\n        cf_inst_to_free=pl.col(\"inst_new\") / (pl.col(\"cf_free_l3\").clip(lower_bound=0) + 1e5),\n        cf_out_to_inc_l3=pl.col(\"cf_out_l3\") / (pl.col(\"cf_inc_l3\") + 1e5),\n        cf_inst_to_minbal=pl.col(\"inst_new\") / (pl.col(\"cf_minbal_l3\") + 1e5),\n    ).drop(\"inst_new\", \"cf_free_l3\", \"cf_inc_l3\")\n    # e-commerce: cancellations, night share, basket size, AOV shift, tightest gap, smooth recency-weighted counts\n    e = tables[\"ecommerce_transactions\"].join(cut.select(\"customer_id\", \"cutoff\"), on=\"customer_id\")\n    e = e.filter(pl.col(\"transaction_datetime\") <= pl.col(\"cutoff\"))\n    o = e.group_by(\"order_id\").agg(pl.col(\"customer_id\", \"cutoff\", \"transaction_datetime\", \"order_total\", \"order_status\").first(),\n                                   n_items=pl.len())\n    o = o.with_columns(age_d=_days(\"cutoff\", \"transaction_datetime\"), done=pl.col(\"order_status\") == \"completed\",\n                       night=pl.col(\"transaction_datetime\").dt.hour() < 5)\n    r28, old = pl.col(\"age_d\") <= 28, pl.col(\"age_d\") > 182\n    ec = o.group_by(\"customer_id\").agg(\n        ec2_base=(old & (pl.col(\"age_d\") <= 730)).sum()\n        / ((pl.col(\"age_d\").max().clip(upper_bound=730) - 182).clip(lower_bound=14) / 7),\n        ec2_cancel_28=(~pl.col(\"done\")).filter(r28).sum(),\n        ec2_night_share_90=pl.col(\"night\").filter(pl.col(\"age_d\") <= 90).mean(),\n        ec2_items_per_order_28=pl.col(\"n_items\").filter(r28).mean(),\n        ec2_aov_28=pl.col(\"order_total\").filter(pl.col(\"done\") & r28).mean(),\n        ec2_aov_old=pl.col(\"order_total\").filter(pl.col(\"done\") & old).mean(),\n        ec2_gap_min_28=pl.col(\"age_d\").filter(r28).sort().diff().min(),\n        **{f\"ec2_ewm{t}\": (-pl.col(\"age_d\") / t).exp().sum() for t in (7, 28, 56)},\n    ).with_columns(\n        ec2_aov_shift=pl.col(\"ec2_aov_28\") / pl.col(\"ec2_aov_old\"),\n        ec2_ewm7_rel=pl.col(\"ec2_ewm7\") / (pl.col(\"ec2_base\") + 0.05),\n        ec2_ewm28_exc=pl.col(\"ec2_ewm28\") - pl.col(\"ec2_base\") * 4,\n        ec2_ewm56_rel=pl.col(\"ec2_ewm56\") / (pl.col(\"ec2_base\") * 8 + 0.05),\n    ).drop(\"ec2_aov_28\", \"ec2_aov_old\", \"ec2_ewm7\", \"ec2_ewm28\", \"ec2_ewm56\")\n    out = cut.select(\"customer_id\")\n    for part in (bz, ob, rq, cf, ec):\n        out = out.join(part, on=\"customer_id\", how=\"left\")\n    return out\n\n\nPP_CATS = [\"groceries\", \"fashion\", \"electronics\", \"home\", \"health\", \"education\", \"personal_care\"]\n\n\ndef v5_features(apps: pl.DataFrame, tables: dict) -> pl.DataFrame:\n    \"\"\"v5 additions (eda/40-45): unit-price premium within category, basket quantity, decision month.\n    Category reference (mean and sd of log unit price, eda/50): an expanding window over item rows of all customers\n    (no labels) dated before the first day of the application's decision month, so every application is normalised\n    only with prices that were already observed when it was decided.\"\"\"\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"), dm=pl.col(\"decision_datetime\").dt.truncate(\"1mo\"))\n    raw = tables[\"ecommerce_transactions\"].with_columns(lu=(pl.col(\"item_total\") / pl.col(\"quantity\")).log(),\n                                                        ym=pl.col(\"transaction_datetime\").dt.truncate(\"1mo\"))\n    ms = raw.group_by(\"product_category\", \"ym\").agg(n=pl.len(), s=pl.col(\"lu\").sum(), ss=(pl.col(\"lu\") ** 2).sum())\n    ref = (cut.select(\"dm\").unique().join(ms, how=\"cross\").filter(pl.col(\"ym\") < pl.col(\"dm\"))\n           .group_by(\"product_category\", \"dm\").agg(pl.col(\"n\", \"s\", \"ss\").sum())\n           .with_columns(mu=pl.col(\"s\") / pl.col(\"n\"),\n                         sd=((pl.col(\"ss\") - pl.col(\"s\") ** 2 / pl.col(\"n\")) / (pl.col(\"n\") - 1)).sqrt()))\n    e = raw.join(cut, on=\"customer_id\").filter(pl.col(\"transaction_datetime\") <= pl.col(\"cutoff\"))\n    e = e.join(ref.select(\"product_category\", \"dm\", \"mu\", \"sd\"), on=[\"product_category\", \"dm\"], how=\"left\")\n    e = e.with_columns(age_d=_days(\"cutoff\", \"transaction_datetime\"), done=pl.col(\"order_status\") == \"completed\",\n                       z=(pl.col(\"lu\") - pl.col(\"mu\")) / pl.col(\"sd\"))\n    lo, hi = pl.col(\"z\").quantile(0.1), pl.col(\"z\").quantile(0.9)\n    pp = e.group_by(\"customer_id\").agg(\n        pp_mean=pl.col(\"z\").mean(),\n        pp_trim=pl.col(\"z\").filter((pl.col(\"z\") > lo) & (pl.col(\"z\") < hi)).mean(),\n        pp_done=pl.col(\"z\").filter(\"done\").mean(), pp_min=pl.col(\"z\").min(), pp_max=pl.col(\"z\").max(),\n        pp_30=pl.col(\"z\").filter(pl.col(\"age_d\") <= 30).mean(), pp_90=pl.col(\"z\").filter(pl.col(\"age_d\") <= 90).mean(),\n        pp_old=pl.col(\"z\").filter(pl.col(\"age_d\") > 182).mean(),\n        pp_qty_item=pl.col(\"quantity\").mean(), pp_qty_item_90=pl.col(\"quantity\").filter(pl.col(\"age_d\") <= 90).mean(),\n        **{f\"pp_cat_{c}\": pl.col(\"z\").filter(pl.col(\"product_category\") == c).mean() for c in PP_CATS},\n    )\n    month = apps.select(\"customer_id\", dec_month=pl.col(\"decision_datetime\").dt.month())\n    return month.join(pp, on=\"customer_id\", how=\"left\")\n\n\ndef build_all(tables: dict) -> pl.DataFrame:\n    \"\"\"tables: dict of polars frames keyed by table name. Returns one row per selected application.\"\"\"\n    tr = tables[\"train_applications\"].with_columns(is_train=pl.lit(1))\n    te = tables[\"test_applications\"].with_columns(target=pl.lit(None, pl.Int64), is_train=pl.lit(0))\n    apps = pl.concat([tr, te], how=\"diagonal_relaxed\")\n    f = apps.select(\"application_id\", \"customer_id\", \"decision_datetime\", \"target\", \"is_train\")\n    for part in [\n        app_features(apps, tables[\"customer_profiles\"]),\n        bureau_features(apps, tables[\"bureau_observations\"]),\n        repayment_features(apps, tables[\"repayment_history\"], tables[\"previous_applications\"]),\n        account_features(apps, tables[\"account_behavior\"]),\n        ecommerce_features(apps, tables[\"ecommerce_transactions\"]),\n        prev_features(apps, tables[\"previous_applications\"]),\n        v2_features(apps, tables),\n        v5_features(apps, tables),\n    ]:\n        f = f.join(part, on=\"customer_id\", how=\"left\")\n    # cross-table ratios: the real affordability picture\n    f = f.with_columns(\n        has_history=pl.col(\"pv_n\").is_not_null().cast(pl.Int8),\n        dti_verified=pl.col(\"proposed_monthly_installment\") / pl.col(\"ac_income_mean\"),\n        dti_total=(pl.col(\"proposed_monthly_installment\") + pl.col(\"rp_active_inst\").fill_null(0))\n        / pl.col(\"declared_monthly_income\"),\n        income_verif_ratio=pl.col(\"ac_income_mean\") / pl.col(\"declared_monthly_income\"),\n        debt_to_income=pl.col(\"bc_outstanding_max\") / pl.col(\"declared_monthly_income\"),\n        inst_to_avgbal=pl.col(\"proposed_monthly_installment\") / (pl.col(\"ac_avgbal_mean\") + 1),\n        ecspend_to_income=pl.col(\"ec_spend_per_month\") / pl.col(\"declared_monthly_income\"),\n    )\n    return f\n", "nn_prep.py": "\"\"\"Neural-network input preparation (TabM). Pure numpy/scipy so it runs and is verified locally (eda/32).\n\nPer numeric column: mid-rank ECDF -> standard-normal map fitted on train rows only (rank-preserving, so univariate\nKS is unchanged; a value shared by many rows, such as 0 days late, lands in the middle of its tie block, not at\nan extreme), clipped to +-5, NaN -> 0 (the median). Columns with an identical missingness pattern share one\nmissing-indicator, because thin-file customers miss whole tables at once. Constant columns are dropped.\"\"\"\nimport numpy as np\nfrom scipy.special import ndtri\n\n\ndef nn_prep(train: np.ndarray, test: np.ndarray):\n    \"\"\"train/test: float arrays (rows x numeric features), NaN/inf allowed. Returns (Xtr, Xte, info), float32.\"\"\"\n    train = np.where(np.isfinite(train), train, np.nan)\n    test = np.where(np.isfinite(test), test, np.nan)\n    ref = [np.sort(train[~np.isnan(train[:, j]), j]) for j in range(train.shape[1])]\n    keep = [j for j, s in enumerate(ref) if len(s) and s[0] < s[-1]]\n\n    def transform(a):\n        out = np.zeros((a.shape[0], len(keep)), np.float32)\n        for i, j in enumerate(keep):\n            s, x = ref[j], a[:, j]\n            m = ~np.isnan(x)\n            u = (np.searchsorted(s, x[m], side=\"left\") + np.searchsorted(s, x[m], side=\"right\")) / (2 * len(s))\n            out[m, i] = np.clip(ndtri(np.clip(u, 0.5 / len(s), 1 - 0.5 / len(s))), -5, 5)\n        return out\n\n    miss_tr, miss_te = np.isnan(train[:, keep]), np.isnan(test[:, keep])\n    seen, ind_cols = set(), []\n    for i in range(miss_tr.shape[1]):\n        key = np.packbits(miss_tr[:, i]).tobytes()\n        if miss_tr[:, i].any() and key not in seen:\n            seen.add(key)\n            ind_cols.append(i)\n    Xtr = np.hstack([transform(train), miss_tr[:, ind_cols].astype(np.float32)])\n    Xte = np.hstack([transform(test), miss_te[:, ind_cols].astype(np.float32)])\n    return Xtr, Xte, {\"kept\": keep, \"indicator_of\": ind_cols}\n", "valid2.py": "\"\"\"validation_v2: one protocol for local screening (one fold) and full Kaggle runs.\n\nOuter: StratifiedKFold(5, shuffle, seed) per seed; the outer validation fold is only scored, never used for any decision.\nInner: a stratified 1/7 split of the outer training fold picks the iteration / epoch (early stopping). The model is then\nrefitted on the whole outer training fold with that fixed budget (GBDT: best x REFIT, TabM: best epoch).\nEvery run writes OOF and test predictions with application_id. Blend weights are chosen later, cross-fitted over the\nouter folds (eda/blend2.py), never on the rows they are scored on.\"\"\"\nimport gc\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport polars as pl\nfrom scipy.stats import ks_2samp\nfrom sklearn.model_selection import StratifiedKFold\n\nID_COLS = [\"application_id\", \"customer_id\", \"decision_datetime\", \"target\", \"is_train\"]\nCAT_COLS = [\"loan_purpose\", \"region\", \"employment_type\", \"housing_status\"]\nN_FOLDS, INNER, REFIT = 5, 7, 1.15\nSEEDS = [42, 7, 2026]\nN_ROUNDS, EARLY_STOP = 10000, 200\nTHREADS = os.cpu_count() or 4\n\nLGB = dict(objective=\"binary\", metric=\"auc\", learning_rate=0.02, num_leaves=31, min_child_samples=100,\n           feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, n_jobs=THREADS)\nLGB_S = dict(LGB, num_leaves=7, max_depth=3, min_child_samples=200, feature_fraction=0.4, lambda_l2=10.0)\nXGB = dict(objective=\"binary:logistic\", eval_metric=\"auc\", learning_rate=0.02, max_depth=5, min_child_weight=5,\n           subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, tree_method=\"hist\", device=\"cuda\")\nXGB_S = dict(XGB, max_depth=3, min_child_weight=20, colsample_bytree=0.4, reg_lambda=10.0)\nCAT = dict(learning_rate=0.03, depth=6, l2_leaf_reg=5, loss_function=\"Logloss\", eval_metric=\"AUC\",\n           task_type=\"GPU\", verbose=0)\nCAT_S = dict(CAT, depth=4, l2_leaf_reg=10)\nTABM = dict(lr=2e-3, wd=3e-4, bins=48, batch=512, epochs=60, patience=8)\n\n\ndef ks(y, p):\n    return float(ks_2samp(p[y == 1], p[y == 0]).statistic)\n\n\ndef outer(y, seed):\n    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))\n\n\ndef inner(y_tr, seed, fold):\n    \"\"\"(fit, stop) positions inside the outer training fold.\"\"\"\n    return next(StratifiedKFold(INNER, shuffle=True, random_state=1000 * seed + fold).split(np.zeros(len(y_tr)), y_tr))\n\n\ndef frames(features: pl.DataFrame):\n    \"\"\"features (build_all output) -> X, Xt (pandas; categoricals typed with train categories), y, train ids, test ids.\"\"\"\n    f = features.sort(\"application_id\")\n    tr, te = f.filter(pl.col(\"is_train\") == 1), f.filter(pl.col(\"is_train\") == 0)\n    cols = sorted(c for c in f.columns if c not in ID_COLS)\n    X, Xt = tr.select(cols).to_pandas(), te.select(cols).to_pandas()\n    for c in cols:\n        if c in CAT_COLS or f[c].dtype == pl.String:\n            cats = sorted(X[c].dropna().unique())\n            X[c], Xt[c] = pd.Categorical(X[c], categories=cats), pd.Categorical(Xt[c], categories=cats)\n        else:\n            X[c] = X[c].astype(float).replace([np.inf, -np.inf], np.nan)\n            Xt[c] = Xt[c].astype(float).replace([np.inf, -np.inf], np.nan)\n    return X, Xt, tr[\"target\"].to_numpy().astype(int), tr[\"application_id\"], te[\"application_id\"]\n\n\n# --- model fits: fit(X, y, tr, fit, stop, rest, seed) -> list of predictions for each frame in rest ---------------\n\ndef lgb_fit(params):\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(params, seed=seed)\n        d_fit = lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]])\n        d_st = lgb.Dataset(X.iloc[tr[st]], y[tr[st]], reference=d_fit)\n        m = lgb.train(p, d_fit, N_ROUNDS, valid_sets=[d_st], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n        n = math.ceil(m.best_iteration * REFIT)\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr], y[tr]), n)\n        return [m.predict(r) for r in rest], n\n    return fit\n\n\ndef xgb_fit(params):\n    import xgboost as xgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(params, seed=seed)\n        d_fit = xgb.DMatrix(X.iloc[tr[fi]], y[tr[fi]], enable_categorical=True)\n        d_st = xgb.DMatrix(X.iloc[tr[st]], y[tr[st]], enable_categorical=True)\n        m = xgb.train(p, d_fit, N_ROUNDS, evals=[(d_st, \"st\")], early_stopping_rounds=EARLY_STOP, verbose_eval=False)\n        n = math.ceil((m.best_iteration + 1) * REFIT)\n        m = xgb.train(p, xgb.DMatrix(X.iloc[tr], y[tr], enable_categorical=True), n)\n        return [m.predict(xgb.DMatrix(r, enable_categorical=True)) for r in rest], n\n    return fit\n\n\ndef cat_fit(params):\n    from catboost import CatBoostClassifier, Pool\n\n    def s(df):\n        out = df.copy()\n        for c in CAT_COLS:\n            out[c] = out[c].astype(str)\n        return out\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        cats = [c for c in CAT_COLS if c in X.columns]\n        m = CatBoostClassifier(**params, iterations=N_ROUNDS, od_type=\"Iter\", od_wait=EARLY_STOP, use_best_model=True,\n                               random_seed=seed)\n        m.fit(Pool(s(X.iloc[tr[fi]]), y[tr[fi]], cat_features=cats), eval_set=Pool(s(X.iloc[tr[st]]), y[tr[st]], cat_features=cats))\n        n = math.ceil((m.get_best_iteration() + 1) * REFIT)\n        m = CatBoostClassifier(**params, iterations=n, random_seed=seed)\n        m.fit(Pool(s(X.iloc[tr]), y[tr], cat_features=cats))\n        return [m.predict_proba(s(r))[:, 1] for r in rest], n\n    return fit\n\n\ndef tabm_fit(cfg=TABM, device=\"cuda\"):\n    import torch\n    import tabm\n    import rtdl_num_embeddings\n    from sklearn.metrics import roc_auc_score\n    from nn_prep import nn_prep\n\n    def arrays(X, rows_list):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        return [X.iloc[r][nums].to_numpy(dtype=float) for r in rows_list]\n\n    def codes(df):\n        return np.column_stack([df[c].cat.codes.to_numpy() + 1 for c in CAT_COLS]).astype(np.int64)\n\n    def train(xn, xc, yv, cards, n_epochs, seed, watch=None):\n        torch.manual_seed(seed)\n        dev = torch.device(device)\n        xn_t, xc_t = torch.as_tensor(xn, device=dev), torch.as_tensor(xc, device=dev)\n        yt = torch.as_tensor(yv, dtype=torch.float32, device=dev)\n        emb = rtdl_num_embeddings.PiecewiseLinearEmbeddings(\n            rtdl_num_embeddings.compute_bins(xn_t, n_bins=cfg[\"bins\"]), d_embedding=16, activation=False, version=\"B\")\n        model = tabm.TabM.make(n_num_features=xn.shape[1], cat_cardinalities=cards, d_out=1, num_embeddings=emb).to(dev)\n        opt = torch.optim.AdamW(model.parameters(), lr=cfg[\"lr\"], weight_decay=cfg[\"wd\"])\n\n        def predict(a, c):\n            model.eval()\n            a, c = torch.as_tensor(a, device=dev), torch.as_tensor(c, device=dev)\n            with torch.no_grad():\n                return torch.cat([torch.sigmoid(model(a[i:i + 1024], c[i:i + 1024]).squeeze(-1).float()).mean(1)\n                                  for i in range(0, len(a), 1024)]).cpu().numpy()\n\n        best, best_ep, wait = -1.0, n_epochs, 0\n        for ep in range(1, n_epochs + 1):\n            model.train()\n            for idx in torch.randperm(len(xn_t), device=dev).split(cfg[\"batch\"]):\n                out = model(xn_t[idx], xc_t[idx]).squeeze(-1)\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(out, yt[idx].unsqueeze(1).expand_as(out))\n                opt.zero_grad()\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                opt.step()\n            if watch is not None:\n                auc = roc_auc_score(watch[2], predict(watch[0], watch[1]))\n                best, best_ep, wait = (auc, ep, 0) if auc > best else (best, best_ep, wait + 1)\n                if wait >= cfg[\"patience\"]:\n                    break\n        return model, predict, best_ep\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        cards = [len(X[c].cat.categories) + 1 for c in CAT_COLS]\n        a_fit, a_st = arrays(X, [tr[fi], tr[st]])\n        x_fit, x_st, _ = nn_prep(a_fit, a_st)\n        _, _, ep = train(x_fit, codes(X.iloc[tr[fi]]), y[tr[fi]], cards, cfg[\"epochs\"], seed,\n                         watch=(x_st, codes(X.iloc[tr[st]]), y[tr[st]]))\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        a_tr = X.iloc[tr][nums].to_numpy(dtype=float)\n        a_rest = np.vstack([r[nums].to_numpy(dtype=float) for r in rest])\n        x_tr, x_rest, _ = nn_prep(a_tr, a_rest)\n        model, predict, _ = train(x_tr, codes(X.iloc[tr]), y[tr], cards, ep, seed)\n        p = predict(x_rest, np.vstack([codes(r) for r in rest]))\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), ep\n    return fit\n\n\ndef ks_nan(y, x):\n    x = np.asarray(x, float)\n    if np.isnan(x).all():\n        return 0.0\n    lo, hi = np.nanmin(x) - 1, np.nanmax(x) + 1\n    return max(ks(y, np.where(np.isnan(x), lo, x)), ks(y, np.where(np.isnan(x), hi, x)))\n\n\ndef fm_select(X, y, tr, n_num=100):\n    \"\"\"v8 recipe: numeric columns ranked by univariate KS on the training fold, greedy |Spearman| < 0.95, top n_num,\n    plus the 4 categoricals.\"\"\"\n    nums = [c for c in X.columns if c not in CAT_COLS]\n    sub = X.iloc[tr]\n    ranked = sorted(nums, key=lambda c: -ks_nan(y[tr], sub[c].to_numpy(dtype=float)))[:2 * n_num]\n    corr = sub[ranked].rank().corr().abs().fillna(0.0)\n    keep = []\n    for c in ranked:\n        if all(corr.loc[c, k] < 0.95 for k in keep):\n            keep.append(c)\n        if len(keep) == n_num:\n            break\n    return keep + CAT_COLS\n\n\ndef icl_fit(make, n_num=100, max_context=60000, chunk=10000, as_codes=False):\n    \"\"\"In-context learners: no early stopping (the inner split is unused). make(seed, cat_idx) -> sklearn-like model.\"\"\"\n    def fit(X, y, tr, fi, st, rest, seed):\n        cols = fm_select(X, y, tr, n_num)\n        ctx = tr if len(tr) <= max_context else np.random.default_rng(seed).choice(tr, max_context, replace=False)\n\n        def frame(d):\n            d = d[cols].reset_index(drop=True)\n            if as_codes:\n                d = d.copy()\n                for c in CAT_COLS:\n                    d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)\n                return d.to_numpy(dtype=np.float32)\n            return d\n        m = make(seed, [cols.index(c) for c in CAT_COLS])\n        m.fit(frame(X.iloc[ctx]), y[ctx])\n        preds = [np.concatenate([np.asarray(m.predict_proba(frame(r.iloc[i:i + chunk])))[:, 1]\n                                 for i in range(0, len(r), chunk)]) for r in rest]\n        import torch\n        del m\n        gc.collect()\n        torch.cuda.empty_cache()\n        return preds, len(cols)\n    return fit\n\n\ndef causilo_fit(n_num=100, n_est=4, max_context=60000, device=\"cuda\"):\n    from causilo import CausiloClassifier\n    return icl_fit(lambda seed, ci: CausiloClassifier(n_estimators=n_est, random_state=seed, device=device),\n                   n_num, max_context)\n\n\ndef tabpfn_fit(n_num=100, n_est=4, max_context=60000, device=\"cuda\"):\n    from tabpfn import TabPFNClassifier\n    return icl_fit(lambda seed, ci: TabPFNClassifier(n_estimators=n_est, random_state=seed, device=device,\n                                                     ignore_pretraining_limits=True, categorical_features_indices=ci,\n                                                     fit_mode=\"fit_with_cache\"),\n                   n_num, max_context, chunk=5000, as_codes=True)\n\n\ndef tabdpt_fit(n_num=100, context=4096, n_ens=8, device=\"cuda\"):\n    \"\"\"TabDPT retrieves `context` nearest training rows per query (its own context reduction).\"\"\"\n    from tabdpt import TabDPTClassifier\n\n    class M:\n        def __init__(self, seed):\n            self.m, self.seed = TabDPTClassifier(device=device, context_reduction=\"retrieval\", missing_indicators=True,\n                                                 compile=False, verbose=False), seed\n\n        def fit(self, Xa, ya):\n            self.m.fit(Xa, ya)\n\n        def predict_proba(self, Xa):\n            return np.mean([self.m.predict_proba(Xa, context_size=context, batch_size=64, seed=self.seed + k) for k in range(n_ens)], axis=0)\n    return icl_fit(lambda seed, ci: M(seed), n_num, 10 ** 9, chunk=2048, as_codes=True)\n\n\ndef realmlp_fit(n_ens=1, **kw):\n    \"\"\"pytabkit RealMLP-TD (tuned defaults). Checkpoint selection only on the inner stop split (X_val), then the\n    selected network predicts the outer fold and test (no refit: RealMLP's schedule is tied to its epoch budget).\"\"\"\n    from pytabkit import RealMLP_TD_Classifier\n\n    def prep(X, rows):\n        \"\"\"Median fill fitted on `rows`, one indicator per distinct missingness pattern (thin-file blocks).\"\"\"\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        med = X.iloc[rows][nums].median()\n        miss = X.iloc[rows][nums].isna()\n        groups, seen = [], set()\n        for c in nums:\n            key = np.packbits(miss[c].to_numpy()).tobytes()\n            if miss[c].any() and key not in seen:\n                seen.add(key)\n                groups.append(c)\n\n        def tf(d):\n            out = d[nums].fillna(med).fillna(0.0)\n            for c in groups:\n                out[f\"na_{c}\"] = d[c].isna().astype(float)\n            for c in CAT_COLS:\n                out[c] = d[c]\n            return out\n        return tf\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        tf = prep(X, tr[fi])\n        X = tf(X)\n        rest = [tf(r) for r in rest]\n        m = RealMLP_TD_Classifier(device=\"cuda\", random_state=seed, n_ens=n_ens, verbosity=0, val_metric_name=\"1-auc_ovr\",\n                                  **kw)\n        m.fit(X.iloc[tr[fi]], y[tr[fi]], X_val=X.iloc[tr[st]], y_val=y[tr[st]], cat_col_names=CAT_COLS)\n        return [m.predict_proba(r)[:, 1] for r in rest], 0\n    return fit\n\n\ndef woe_lr_fit(n_bins=20, alpha=20.0, Cs=(0.003, 0.01, 0.03, 0.1, 0.3)):\n    \"\"\"Credit scorecard: per feature, bins fitted on the training rows (NaN and exact zero get their own bins, else\n    quantile bins of the rest; categoricals by level), smoothed WOE = log odds of the bin shrunk to the prior with\n    `alpha` pseudo-rows, then L2 logistic regression on the standardised WOE columns. C is picked on the inner stop\n    split, then the LR is refitted on the whole training fold (binning refitted there too).\"\"\"\n    from sklearn.linear_model import LogisticRegression\n\n    def encoder(X, rows, yv):\n        prior = yv.mean()\n        lo = np.log(prior / (1 - prior))\n        specs = []\n        for c in X.columns:\n            if c in CAT_COLS:\n                codes = X[c].cat.codes.to_numpy()\n                specs.append((c, \"cat\", None))\n                b = codes[rows]\n            else:\n                x = X[c].to_numpy(dtype=float)[rows]\n                nz = np.isfinite(x) & (x != 0)\n                edges = np.unique(np.quantile(x[nz], np.linspace(0, 1, n_bins + 1))[1:-1]) if nz.sum() > n_bins else np.array([])\n                specs.append((c, \"num\", edges))\n                b = bins_of(x, edges)\n            keys, inv = np.unique(b, return_inverse=True)\n            pos = np.bincount(inv, weights=yv, minlength=len(keys))\n            n = np.bincount(inv, minlength=len(keys))\n            rate = (pos + alpha * prior) / (n + alpha)\n            specs[-1] = specs[-1] + (dict(zip(keys.tolist(), (np.log(rate / (1 - rate)) - lo).tolist())),)\n        return specs\n\n    def bins_of(x, edges):\n        out = np.where(np.isnan(x), -2, np.where(x == 0, -1, np.searchsorted(edges, x, side=\"right\")))\n        return out\n\n    def transform(specs, D):\n        cols = []\n        for c, kind, edges, table in specs:\n            b = D[c].cat.codes.to_numpy() if kind == \"cat\" else bins_of(D[c].to_numpy(dtype=float), edges)\n            cols.append(np.array([table.get(int(k), 0.0) for k in b]) if len(table) < 50 else\n                        np.vectorize(lambda k: table.get(int(k), 0.0))(b))\n        return np.column_stack(cols)\n\n    def fit_lr(A, yv, C, seed):\n        mu, sd = A.mean(0), A.std(0) + 1e-9\n        m = LogisticRegression(C=C, max_iter=2000, random_state=seed).fit((A - mu) / sd, yv)\n        return lambda B: m.decision_function((B - mu) / sd)\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        sp = encoder(X, tr[fi], y[tr[fi]])\n        A, S = transform(sp, X.iloc[tr[fi]]), transform(sp, X.iloc[tr[st]])\n        best = max(Cs, key=lambda C: ks(y[tr[st]], fit_lr(A, y[tr[fi]], C, seed)(S)))\n        sp = encoder(X, tr, y[tr])\n        f = fit_lr(transform(sp, X.iloc[tr]), y[tr], best, seed)\n        return [1 / (1 + np.exp(-f(transform(sp, r)))) for r in rest], best\n    return fit\n\n\ndef lr_q_fit(Cs=(0.001, 0.003, 0.01, 0.03, 0.1, 1.0), onehot=True):\n    \"\"\"Logistic regression on nn_prep inputs (mid-rank ECDF -> normal fitted on the training rows, NaN -> 0, one\n    indicator per missingness pattern) + one-hot categoricals. C picked on the inner stop split, refit on the fold.\"\"\"\n    from sklearn.linear_model import LogisticRegression\n    from nn_prep import nn_prep\n\n    def mats(X, fit_rows, others):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        a, b, _ = nn_prep(X.iloc[fit_rows][nums].to_numpy(dtype=float),\n                          np.vstack([o[nums].to_numpy(dtype=float) for o in others]))\n        def oh(d):\n            return np.column_stack([(d[c].cat.codes.to_numpy()[:, None] == np.arange(len(d[c].cat.categories))).astype(np.float32)\n                                    for c in CAT_COLS])\n        if onehot:\n            a = np.hstack([a, oh(X.iloc[fit_rows])])\n            b = np.hstack([b, np.vstack([oh(o) for o in others])])\n        return a, np.split(b, np.cumsum([len(o) for o in others])[:-1])\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        a, (s_,) = mats(X, tr[fi], [X.iloc[tr[st]]])\n        best = max(Cs, key=lambda C: ks(y[tr[st]], LogisticRegression(C=C, max_iter=3000).fit(a, y[tr[fi]]).decision_function(s_)))\n        a, outs = mats(X, tr, rest)\n        m = LogisticRegression(C=best, max_iter=3000).fit(a, y[tr])\n        return [m.predict_proba(o)[:, 1] for o in outs], best\n    return fit\n\n\ndef run(name, fit, X, Xt, y, ids, test_ids, out: Path, seeds=SEEDS, folds=None, deadline=None):\n    \"\"\"Runs fit over seeds x outer folds. OOF = mean over seeds (NaN where no fold ran); test = mean over all fits.\n    Writes out/oof_<name>.parquet, test_<name>.parquet and log_<name>.csv.\"\"\"\n    out.mkdir(parents=True, exist_ok=True)\n    oof_sum, oof_n, test_sum, log = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(Xt)), []\n    for seed in seeds:\n        for k, (tr, va) in enumerate(outer(y, seed)):\n            if folds is not None and k not in folds:\n                continue\n            if deadline is not None and time.time() > deadline:\n                raise TimeoutError(f\"{name}: deadline before seed {seed} fold {k}\")\n            t0 = time.time()\n            fi, st = inner(y[tr], seed, k)\n            (pv, pt), budget = fit(X, y, tr, fi, st, [X.iloc[va], Xt], seed)\n            oof_sum[va] += pv\n            oof_n[va] += 1\n            test_sum += pt\n            log.append(dict(model=name, seed=seed, fold=k, ks=ks(y[va], pv), budget=budget, sec=round(time.time() - t0, 1)))\n            print(log[-1], flush=True)\n            pd.DataFrame(log).to_csv(out / f\"log_{name}.csv\", index=False)\n    oof = np.where(oof_n > 0, oof_sum / np.maximum(oof_n, 1), np.nan)\n    pl.DataFrame({\"application_id\": ids, \"target\": y, name: oof}).write_parquet(out / f\"oof_{name}.parquet\")\n    pl.DataFrame({\"application_id\": test_ids, name: test_sum / len(log)}).write_parquet(out / f\"test_{name}.parquet\")\n    m = oof_n > 0\n    print(f\"{name}: OOF KS {ks(y[m], oof[m]):.5f} on {m.sum()} rows\", flush=True)\n    return oof, test_sum / len(log), log\n\n\ndef manifest(out: Path, extra=None):\n    import importlib.metadata as md\n    pk = {}\n    for n in [\"numpy\", \"pandas\", \"polars\", \"scikit-learn\", \"scipy\", \"lightgbm\", \"xgboost\", \"catboost\", \"torch\", \"tabm\",\n              \"rtdl_num_embeddings\", \"causilo\"]:\n        try:\n            pk[n] = md.version(n)\n        except md.PackageNotFoundError:\n            pk[n] = None\n    cfg = dict(N_FOLDS=N_FOLDS, INNER=INNER, REFIT=REFIT, SEEDS=SEEDS, N_ROUNDS=N_ROUNDS, EARLY_STOP=EARLY_STOP,\n               LGB=LGB, LGB_S=LGB_S, XGB=XGB, XGB_S=XGB_S, CAT=CAT, CAT_S=CAT_S, TABM=TABM)\n    (out / \"manifest.json\").write_text(json.dumps(dict(packages=pk, config=cfg, **(extra or {})), indent=2, default=str))\n", "zoo_screen.py": "\"\"\"TabArena model zoo on the validation_v2 fold-0 screen (seed 42, outer fold 0). Runs in .venv-zoo (tabarena +\nautogluon). Each model is fitted by AutoGluon on the inner-fit rows with tuning_data = the inner 1/7 stop rows (any\nearly stopping / checkpointing sees only those), default TabArena hyperparameters, no bagging, no weighted ensemble.\nThe outer fold is only predicted. Output: results/screen/zoo_f0/oof_<name>.parquet (NaN outside fold 0), log.csv.\nUsage: .venv-zoo/bin/python eda/zoo_screen.py LimiX TabFM Mitra ... [--minutes 40]\"\"\"\nimport argparse\nimport importlib\nimport shutil\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport polars as pl\n\nsys.path.insert(0, str(Path(__file__).parent))\nimport valid2 as v  # noqa: E402\n\nROOT = Path(__file__).resolve().parent.parent\nOUT = ROOT / \"results\" / \"screen\" / \"zoo_f0\"\nTA = {n: \"tabarena.models\" for n in [\"ChimeraBoostModel\", \"EXAONETabularModel\", \"ILTMModel\", \"KNNNewModel\", \"LimiXModel\",\n                                     \"ModernNCAModel\", \"NoriModel\", \"Nori30MModel\", \"OrionMSPModel\",\n                                     \"PerpetualBoosterModel\", \"RealMLPModel\", \"RealTabPFNv25Model\", \"TabPFNv26Model\",\n                                     \"SAPRPTOSSModel\", \"TabDPTModel\", \"TabDPTTurboModel\", \"TabFMModel\", \"TabICLModel\",\n                                     \"TabICLv2Model\", \"TabMModel\", \"TabPFN3Model\", \"TabPFNWideModel\", \"TabSTARModel\",\n                                     \"TabSwiftModel\", \"XRFMModel\"]}\nAG = {\"MitraModel\": \"autogluon.tabular.models\", \"EBMModel\": \"autogluon.tabular.models\",\n      \"NNFastAiTabularModel\": \"autogluon.tabular.models\", \"RFModel\": \"autogluon.tabular.models\",\n      \"XTModel\": \"autogluon.tabular.models\", \"LinearModel\": \"autogluon.tabular.models\",\n      \"TabularNeuralNetTorchModel\": \"autogluon.tabular.models\", \"CatBoostModel\": \"autogluon.tabular.models\",\n      \"LGBModel\": \"autogluon.tabular.models\", \"XGBoostModel\": \"autogluon.tabular.models\"}\n\n\ndef cls(name):\n    mod = {**TA, **AG}[name]\n    return getattr(importlib.import_module(mod), name)\n\n\ndef main(names, minutes, features=None, out=None, max_train=None, hp=None):\n    from autogluon.tabular import TabularPredictor\n    global OUT\n    OUT = Path(out) if out else OUT\n    X, Xt, y, ids, tids = v.frames(pl.read_parquet(features or ROOT / \"data_pq\" / \"features.parquet\"))\n    tr, va = v.outer(y, 42)[0]\n    fi, st = v.inner(y[tr], 42, 0)\n    if max_train and len(fi) > max_train:\n        # shortcut: stratified subsample of the inner-fit rows for models that cannot hold the full context\n        rng = np.random.default_rng(0)\n        keep = np.sort(np.concatenate([rng.choice(fi[y[tr[fi]] == c], int(max_train * (y[tr[fi]] == c).mean()), replace=False)\n                                       for c in (0, 1)]))\n        fi = keep\n    df = X.assign(target=y)\n    OUT.mkdir(parents=True, exist_ok=True)\n    log_path = OUT / \"log.csv\"\n    for name in names:\n        t0 = time.time()\n        path = Path(\"/tmp\") / f\"ag_zoo_{name}\"\n        row = dict(model=name, n_fit=len(fi))\n        try:\n            pred = TabularPredictor(label=\"target\", eval_metric=\"roc_auc\", path=str(path), verbosity=1).fit(\n                train_data=df.iloc[tr[fi]], tuning_data=df.iloc[tr[st]], hyperparameters={cls(name): dict(hp or {})},\n                num_gpus=1, fit_weighted_ensemble=False, time_limit=minutes * 60, raise_on_no_models_fitted=True)\n            p = pred.predict_proba(X.iloc[va])[1].to_numpy()\n            oof = np.full(len(y), np.nan)\n            oof[va] = p\n            pl.DataFrame({\"application_id\": ids, \"target\": y, name: oof}).write_parquet(OUT / f\"oof_{name}.parquet\")\n            row.update(ks=v.ks(y[va], p), status=\"ok\")\n        except Exception as e:  # noqa: BLE001 - a failing model is a result to log, not a reason to stop the zoo\n            traceback.print_exc()\n            row.update(ks=np.nan, status=f\"{type(e).__name__}: {str(e)[:160]}\")\n        row[\"sec\"] = round(time.time() - t0)\n        print(row, flush=True)\n        old = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()\n        pd.concat([old, pd.DataFrame([row])]).to_csv(log_path, index=False)\n        shutil.rmtree(path, ignore_errors=True)\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"names\", nargs=\"+\")\n    ap.add_argument(\"--minutes\", type=float, default=40)\n    ap.add_argument(\"--features\")\n    ap.add_argument(\"--out\")\n    ap.add_argument(\"--max-train\", type=int)\n    ap.add_argument(\"--hp\", default=\"{}\", help=\"JSON hyperparameters for every model\")\n    a = ap.parse_args()\n    import json\n    main(a.names, a.minutes, a.features, a.out, a.max_train, json.loads(a.hp))\n"}
CODE = Path('/kaggle/working/code'); CODE.mkdir(parents=True, exist_ok=True)
for n, s in FILES.items():
    (CODE / n).write_text(s)
sys.path.insert(0, str(CODE))
from features import build_all
import torch; print(torch.__version__, torch.cuda.device_count())

In [ ]:
data = next(Path('/kaggle/input').rglob('train_applications.csv')).parent
names = ['train_applications', 'test_applications', 'customer_profiles', 'previous_applications',
         'repayment_history', 'account_behavior', 'bureau_observations', 'ecommerce_transactions']
tables = {n: pl.read_csv(data / f'{n}.csv', try_parse_dates=True, infer_schema_length=100000) for n in names}
OUT = Path('/kaggle/working/v23'); OUT.mkdir(exist_ok=True)
feats = build_all(tables); feats.write_parquet(OUT / 'features.parquet'); print(feats.shape); del tables, feats

In [ ]:
JOBS = [[[0, "LimiXModel", []], [0, "TabICLv2Model TabICLModel", []], [0, "ILTMModel", []]], [[1, "TabDPTModel TabDPTTurboModel", []], [1, "NoriModel Nori30MModel", []], [1, "MitraModel", ["--max-train", "8000"]]]]
def cmd(models, extra, i):
    return ' '.join(map(repr, [sys.executable, str(CODE / 'zoo_screen.py'), *models.split(), '--minutes', '50',
                                '--features', str(OUT / 'features.parquet'), '--out', str(OUT / f'zoo{i}'), *extra]))
procs = []
for i, chain in enumerate(JOBS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(chain[0][0]))
    procs.append(subprocess.Popen(' ; '.join(cmd(m, e, i) for _, m, e in chain), shell=True, env=env, cwd=CODE,
                                  stdout=open(OUT / f'job{i}.log', 'w'), stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(60)
for i in range(len(JOBS)):
    print(open(OUT / f'zoo{i}' / 'log.csv').read() if (OUT / f'zoo{i}' / 'log.csv').exists() else f'job {i}: no log')